# Testing FMP get_historical_prices Method

This notebook tests the `get_historical_prices` method in isolation with a small list of tickers.

## Setup
- Load environment variables
- Import required libraries
- Define FMPClient class with get_historical_prices method


In [1]:
# Import required libraries
import os
import time
import requests
import pandas as pd
from typing import List, Dict, Optional, Tuple
from requests.exceptions import HTTPError, RequestException, Timeout
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("Libraries imported successfully!")
print(f"FMP_API_KEY loaded: {'Yes' if os.getenv('FMP_API_KEY') else 'No'}")


Libraries imported successfully!
FMP_API_KEY loaded: Yes


## FMPClient Class Definition


In [2]:
class FMPClient:
    """Client for interacting with the Financial Modeling Prep API."""
    
    def __init__(self, api_key: Optional[str] = None):
        """
        Initialize the FMP API client.
        
        Args:
            api_key: FMP API key. If not provided, will attempt to load from environment.
            
        Raises:
            ValueError: If API key is not provided and not found in environment.
        """
        self.api_key = api_key or os.getenv("FMP_API_KEY")
        if not self.api_key:
            raise ValueError("FMP_API_KEY environment variable not set. Please set it in your .env file or environment.")
        
        self.base_url = "https://financialmodelingprep.com/api/v3"
        self.session = requests.Session()
        self.session.headers.update({"Content-Type": "application/json"})
    
    def _make_request(self, endpoint: str, params: Optional[Dict] = None) -> Tuple[Optional[Dict], Optional[str]]:
        """
        Make a request to the FMP API with error handling.
        
        Args:
            endpoint: API endpoint path (without base URL)
            params: Optional query parameters
            
        Returns:
            Tuple of (JSON response as dictionary, error_message)
            Returns (None, error_message) if request fails
            Returns (data, None) if request succeeds
        """
        url = f"{self.base_url}/{endpoint}"
        request_params = {"apikey": self.api_key}
        if params:
            request_params.update(params)
        
        try:
            response = self.session.get(url, params=request_params, timeout=30)
            response.raise_for_status()
            return response.json(), None
        except HTTPError as e:
            error_msg = f"HTTP {response.status_code} error for {endpoint}: {e}"
            try:
                error_body = response.text
                error_msg += f"\nResponse body: {error_body}"
            except:
                pass
            return None, error_msg
        except Timeout:
            return None, f"Request timeout for {endpoint}"
        except RequestException as e:
            return None, f"Request error for {endpoint}: {e}"
        except Exception as e:
            return None, f"Unexpected error for {endpoint}: {e}"
    
    def get_historical_prices(self, tickers: List[str], days: int = 252) -> pd.DataFrame:
        """
        Batch fetch historical close prices for the last N trading days.
        
        Uses batch endpoint when possible, with rate limiting to avoid API limits.
        
        Args:
            tickers: List of stock tickers to fetch data for
            days: Number of trading days to fetch (default: 252, approximately 1 year)
            
        Returns:
            pandas DataFrame with dates as index and tickers as columns.
            Each column contains the closing price for that ticker.
            
        Note:
            This method attempts to fetch data for all tickers. If some fail,
            they will be excluded from the result DataFrame.
        """
        if not tickers:
            return pd.DataFrame()
        
        all_data = {}
        failed_tickers = []
        error_details = {}  # Store error details for each failed ticker
        
        # Try batch endpoint first - FMP supports comma-separated tickers
        # Batch size limited to avoid URL length issues (typically 5-10 tickers per batch)
        batch_size = 5
        ticker_batches = [tickers[i:i + batch_size] for i in range(0, len(tickers), batch_size)]
        
        print(f"Processing {len(tickers)} tickers in {len(ticker_batches)} batches...")
        
        for batch_idx, batch in enumerate(ticker_batches, 1):
            try:
                # Try batch endpoint with comma-separated tickers
                ticker_list = ",".join(batch)
                endpoint = f"historical-price-full/{ticker_list}"
                params = {"timeseries": days}
                
                print(f"Batch {batch_idx}/{len(ticker_batches)}: Requesting {ticker_list}...")
                data, error = self._make_request(endpoint, params)
                
                if data is None:
                    # If batch fails, fall back to individual requests for this batch
                    batch_error = error or "Unknown error"
                    print(f"  Batch request failed: {batch_error}")
                    print(f"  Falling back to individual requests for this batch...")
                    for ticker in batch:
                        time.sleep(0.1)  # Small delay to avoid rate limits
                        endpoint = f"historical-price-full/{ticker}"
                        params = {"timeseries": days}
                        ticker_data, ticker_error = self._make_request(endpoint, params)
                        
                        if ticker_data and "historical" in ticker_data:
                            historical = ticker_data.get("historical", [])
                            if historical:
                                df = pd.DataFrame(historical)
                                if "date" in df.columns and "close" in df.columns:
                                    df["date"] = pd.to_datetime(df["date"])
                                    df = df.set_index("date").sort_index()
                                    all_data[ticker] = df["close"]
                                    print(f"    ✓ {ticker}: Success")
                                    continue
                        failed_tickers.append(ticker)
                        if ticker_error:
                            error_details[ticker] = ticker_error
                        print(f"    ✗ {ticker}: Failed")
                    continue
                
                # Process batch response - can be a list or dict
                if isinstance(data, list):
                    # Multiple tickers returned as list
                    for ticker_data in data:
                        if isinstance(ticker_data, dict) and "symbol" in ticker_data:
                            ticker = ticker_data["symbol"]
                            historical = ticker_data.get("historical", [])
                            if historical:
                                df = pd.DataFrame(historical)
                                if "date" in df.columns and "close" in df.columns:
                                    df["date"] = pd.to_datetime(df["date"])
                                    df = df.set_index("date").sort_index()
                                    all_data[ticker] = df["close"]
                                    print(f"  ✓ {ticker}: Success")
                                else:
                                    failed_tickers.append(ticker)
                                    print(f"  ✗ {ticker}: Missing date/close columns")
                            else:
                                failed_tickers.append(ticker)
                                print(f"  ✗ {ticker}: No historical data")
                elif isinstance(data, dict) and "historical" in data:
                    # Single ticker response
                    ticker = batch[0]  # Assume first ticker in batch
                    historical = data.get("historical", [])
                    if historical:
                        df = pd.DataFrame(historical)
                        if "date" in df.columns and "close" in df.columns:
                            df["date"] = pd.to_datetime(df["date"])
                            df = df.set_index("date").sort_index()
                            all_data[ticker] = df["close"]
                            print(f"  ✓ {ticker}: Success")
                        else:
                            failed_tickers.append(ticker)
                            print(f"  ✗ {ticker}: Missing date/close columns")
                    else:
                        failed_tickers.append(ticker)
                        print(f"  ✗ {ticker}: No historical data")
                else:
                    # Unexpected format, fall back to individual
                    print(f"  Unexpected response format, falling back to individual requests...")
                    for ticker in batch:
                        failed_tickers.append(ticker)
                
                # Rate limiting: small delay between batches
                time.sleep(0.2)
                
            except Exception as e:
                error_msg = f"Error processing batch {batch}: {e}"
                print(f"  ✗ Exception: {error_msg}")
                for ticker in batch:
                    failed_tickers.append(ticker)
                    error_details[ticker] = error_msg
        
        if failed_tickers:
            print(f"\n{'='*80}")
            print(f"Warning: Failed to fetch data for {len(failed_tickers)} tickers")
            print(f"{'='*80}")
            
            # Print first few error details
            print("\nSample error details (first 10 failures):")
            for i, ticker in enumerate(failed_tickers[:10]):
                error = error_details.get(ticker, "No error details available")
                print(f"  {ticker}: {error}")
            
            if len(failed_tickers) > 10:
                print(f"\n... and {len(failed_tickers) - 10} more failures")
            
            # Check if there's a common error pattern
            if error_details:
                error_counts = {}
                for error in error_details.values():
                    # Extract status code if present
                    if "HTTP" in error:
                        status_match = error.split("HTTP")[1].split()[0] if "HTTP" in error else None
                        if status_match:
                            error_counts[status_match] = error_counts.get(status_match, 0) + 1
                
                if error_counts:
                    print(f"\nError summary:")
                    for status, count in error_counts.items():
                        print(f"  {status}: {count} failures")
            
            print(f"{'='*80}\n")
        
        if not all_data:
            raise RuntimeError("Failed to fetch historical prices for any tickers")
        
        # Combine all series into a DataFrame
        result_df = pd.DataFrame(all_data)
        result_df.index.name = "date"
        
        return result_df

print("FMPClient class defined successfully!")


FMPClient class defined successfully!


## Test with Small List of Tickers

Let's test with a small list of popular stocks.


In [3]:
# Initialize client
client = FMPClient()

# Test with a small list of tickers
test_tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"]

print(f"Testing with {len(test_tickers)} tickers: {test_tickers}")
print("-" * 80)


Testing with 5 tickers: ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']
--------------------------------------------------------------------------------


In [4]:
# Fetch historical prices (252 trading days = ~1 year)
try:
    df = client.get_historical_prices(test_tickers, days=252)
    
    print("\n" + "="*80)
    print("SUCCESS! Data fetched successfully")
    print("="*80)
    print(f"\nDataFrame shape: {df.shape}")
    print(f"Date range: {df.index.min()} to {df.index.max()}")
    print(f"Tickers retrieved: {list(df.columns)}")
    
    # Display first few rows
    print("\nFirst 5 rows:")
    print(df.head())
    
    # Display last few rows
    print("\nLast 5 rows:")
    print(df.tail())
    
except Exception as e:
    print(f"\nERROR: {e}")
    import traceback
    traceback.print_exc()


Processing 5 tickers in 1 batches...
Batch 1/1: Requesting AAPL,MSFT,GOOGL,AMZN,TSLA...
  Unexpected response format, falling back to individual requests...


Sample error details (first 10 failures):
  AAPL: No error details available
  MSFT: No error details available
  GOOGL: No error details available
  AMZN: No error details available
  TSLA: No error details available


ERROR: Failed to fetch historical prices for any tickers


Traceback (most recent call last):
  File "C:\Users\cadaa\AppData\Local\Temp\ipykernel_3060\4104664109.py", line 3, in <module>
    df = client.get_historical_prices(test_tickers, days=252)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cadaa\AppData\Local\Temp\ipykernel_3060\3168539883.py", line 213, in get_historical_prices
    raise RuntimeError("Failed to fetch historical prices for any tickers")
RuntimeError: Failed to fetch historical prices for any tickers


## Visualize Results (Optional)

Plot the price data if matplotlib is available.


In [ ]:
# Optional: Plot the data if matplotlib is available
try:
    import matplotlib.pyplot as plt
    
    if not df.empty:
        # Normalize prices to start at 100 for comparison
        normalized_df = (df / df.iloc[0] * 100)
        
        plt.figure(figsize=(12, 6))
        for col in normalized_df.columns:
            plt.plot(normalized_df.index, normalized_df[col], label=col, linewidth=2)
        
        plt.title('Normalized Stock Prices (Base = 100)', fontsize=14, fontweight='bold')
        plt.xlabel('Date', fontsize=12)
        plt.ylabel('Normalized Price', fontsize=12)
        plt.legend(loc='best')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        print("Chart displayed successfully!")
    else:
        print("No data to plot.")
        
except ImportError:
    print("Matplotlib not available. Skipping visualization.")
except Exception as e:
    print(f"Error creating plot: {e}")


## Test with Different Batch Sizes

You can modify the batch_size in the method or test with different numbers of tickers.


In [ ]:
# Test with more tickers if desired
# Uncomment and modify as needed:

# more_tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA", "META", "NVDA", "JPM", "V", "JNJ"]
# df_larger = client.get_historical_prices(more_tickers, days=252)
# print(f"\nRetrieved data for {len(df_larger.columns)} tickers")
